# Demo — end-to-end inference
Task 3 fusion tag prediction on one MTAT clip, and Task 4 caption→audio retrieval.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
import torch, numpy as np
from utils import load_config, set_seed
cfg = load_config(); set_seed(cfg['seed'])
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = Path(cfg['paths']['checkpoints']); dev

device(type='cuda')

## Task 3: tag prediction on one MTAT test clip

In [2]:
from fusion_model import FusionModel
from bert_encoder import get_tokenizer
from data_loaders import MTATDataset, make_collate
from torch.utils.data import DataLoader
tags = json.loads((Path(cfg['paths']['data_processed'])/'mtat_labels.json').read_text())['tags']
tok = get_tokenizer(cfg['text']['bert_model'])
ds = MTATDataset(cfg, 'test')
dl = DataLoader(ds, batch_size=1, collate_fn=make_collate(tok, cfg['text']['max_len_metadata']))
model = FusionModel(cfg, len(tags), 'cross_attn').to(dev)
model.load_state_dict(torch.load(ckpt/'task3_cross_attn.pt', map_location=dev)); model.eval()
g, ids, mask, y, cids = next(iter(dl))
with torch.no_grad():
    prob = torch.sigmoid(model(g.to(dev), ids.to(dev), mask.to(dev)))[0].cpu().numpy()
true = [tags[j] for j in np.where(y[0].numpy()>0.5)[0]]
pred = [(tags[j], round(float(prob[j]),3)) for j in np.argsort(-prob)[:5]]
print('clip:', cids[0], '| text:', ds.text.get(str(cids[0]),''))
print('true tags :', true)
print('top-5 pred:', pred)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


C:\Users\user5\AppData\Local\Temp\ipykernel_22696\1403217934.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(ckpt/'task3_cross_attn.pt'

clip: 2 | text: American Bach Soloists - BWV54 - I Aria (J.S. Bach Solo Cantatas)
true tags : ['classical', 'strings', 'violin', 'opera']
top-5 pred: [('opera', 0.91), ('classical', 0.62), ('violin', 0.511), ('strings', 0.416), ('vocal', 0.206)]


## Task 4: caption -> audio retrieval

In [3]:
from contrastive import DualEncoder
from data_loaders import MusicCapsDataset
mc = MusicCapsDataset(cfg, 'test')
dl2 = DataLoader(mc, batch_size=32, collate_fn=make_collate(tok, cfg['text']['max_len_caption']))
de = DualEncoder(cfg).to(dev)
de.load_state_dict(torch.load(ckpt/'task4.pt', map_location=dev)); de.eval()
za, zt, ytids = [], [], []
with torch.no_grad():
    for g, i_ids, m, _, bids in dl2:
        za.append(de.encode_graph(g.to(dev)).cpu()); zt.append(de.encode_text(i_ids.to(dev), m.to(dev)).cpu()); ytids += list(bids)
za, zt = torch.cat(za), torch.cat(zt)
qi = 0
sims = (zt[qi] @ za.t())
top = sims.argsort(descending=True)[:3].tolist()
print('query caption:', mc.caption.get(ytids[qi],'')[:150])
for r, ai in enumerate(top, 1):
    mark = ' <-- correct' if ytids[ai]==ytids[qi] else ''
    print(f'  {r}. {ytids[ai]} sim={sims[ai]:.3f}{mark}  {mc.caption.get(ytids[ai],"")[:90]}')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


C:\Users\user5\AppData\Local\Temp\ipykernel_22696\2876564560.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  de.load_state_dict(torch.load(ckpt/'task4.pt', map_location=

query caption: The Pop song features a soft female vocal singing over sustained pulsating synth lead, mellow piano melody, sustained synth brass, punchy kick, claps.
  1. ALVS3Q_jNaU sim=0.672  This is the recording of a jazz reggae concert. There is a saxophone lead playing a solo. 
  2. HfzEa06vDLg sim=0.653  The female voice is singing lightly sad-sounding. Backing voices are supporting her in som
  3. eWwWwoQLtVg sim=0.618  This is a lullaby piece. There is a female voice singing softly with accentuations at spec
